# 02 · Estatísticas descritivas — Nascidos vivos (SINASC/CE)

Calcula, para as variáveis quantitativas: **medidas de posição** (média, mediana, moda, quartis, percentis), **dispersão** (amplitude, AIQ, variância,
desvio padrão, erro padrão, coeficiente de variação), **forma** (assimetria e curtose) e **valores atípicos** (outliers). Também traz distribuições de
frequência, tabelas de variáveis categóricas e um texto-modelo para o relatório.

Tem q ter rodado o notebook `01_preparacao_dados`.

In [16]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

CANDIDATOS = [
    Path("../../data/processed/Descritiva/SINASC_CE_PADRÃO_OURO_tratado.csv"),
]
CAMINHO = next((p for p in CANDIDATOS if p.exists()), None)
if CAMINHO is None:
    raise FileNotFoundError("Base tratada não encontrada. Rode o notebook 01 primeiro.")

df = pd.read_csv(
    CAMINHO,
    dtype={"CONTADOR": str, "CODMUNNASC": str, "CODMUNRES": str},
    parse_dates=["DTNASC"],
)

PASTA_TABELAS = Path("../outputs/tabelas")
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)

VARIAVEIS = {
    "PESO": "Peso ao nascer (g)",
    "IDADEMAE": "Idade da mãe (anos)",
    "SEMAGESTAC": "Semanas de gestação",
    "CONSPRENAT": "Consultas de pré-natal",
    "APGAR1": "Apgar 1º minuto",
    "APGAR5": "Apgar 5º minuto",
    "QTDGESTANT": "Gestações anteriores",
    "QTDFILVIVO": "Filhos vivos anteriores",
    "QTDFILMORT": "Filhos mortos anteriores",
    "QTDPARTNOR": "Partos vaginais anteriores",
    "QTDPARTCES": "Partos cesáreos anteriores",
    "IDADEPAI": "Idade do pai (anos)",
}
print("Base:", df.shape, "| variáveis analisadas:", len(VARIAVEIS))

Base: (142568, 28) | variáveis analisadas: 12


## 1. Funções de cálculo

| Medida | Fórmula / definição |
|---|---|
| Média | $\bar{x} = \frac{1}{n}\sum x_i$ |
| Variância amostral | $s^2 = \frac{\sum (x_i-\bar{x})^2}{n-1}$ |
| Desvio padrão | $s = \sqrt{s^2}$ |
| Erro padrão da média | $EP = s/\sqrt{n}$ |
| Coeficiente de variação | $CV = \frac{s}{\bar{x}}\times 100\%$ |
| Amplitude interquartílica (AIQ) | $AIQ = Q_3 - Q_1$ |
| Limites de Tukey (outliers) | $Q_1 - 1{,}5\cdot AIQ$ e $Q_3 + 1{,}5\cdot AIQ$ |
| IC 95% da média | $\bar{x} \pm 1{,}96\cdot EP$ (válido para n grande) |

**Convenções adotadas** (variam entre autores — cite a que usar no relatório):
- **CV:** < 15% baixa dispersão · 15–30% média · > 30% alta.
- **Assimetria (g₁):** |g₁| < 0,5 aproximadamente simétrica · 0,5–1 moderada · > 1 forte (positiva = cauda à direita).
- **Curtose (excesso):** ≈ 0 mesocúrtica · > 0 leptocúrtica (caudas pesadas/pico alto) · < 0 platicúrtica.
- **Quartis:** interpolação linear (pandas; equivale ao `QUARTIL.INC` do Excel). Outros métodos dão diferenças pequenas.
- `skew()` e `kurt()` do pandas usam os estimadores amostrais ajustados (iguais ao `DISTORÇÃO` e `CURT` do Excel).

In [17]:
def classificar_cv(cv):
    if np.isnan(cv):
        return "—"
    if cv < 15:
        return "Baixa dispersão"
    if cv <= 30:
        return "Média dispersão"
    return "Alta dispersão"


def classificar_assimetria(g):
    if abs(g) < 0.5:
        return "Aproximadamente simétrica"
    lado = "à direita (positiva)" if g > 0 else "à esquerda (negativa)"
    forca = "moderada" if abs(g) < 1 else "forte"
    return "Assimetria " + forca + " " + lado


def classificar_curtose(k):
    if abs(k) < 0.5:
        return "Mesocúrtica"
    return "Leptocúrtica" if k > 0 else "Platicúrtica"


def calcular_estatisticas(serie):
    total = len(serie)
    s = serie.dropna().astype(float)
    n = int(s.size)
    media = s.mean()
    dp = s.std(ddof=1)
    q1, mediana, q3 = s.quantile([0.25, 0.5, 0.75])
    aiq = q3 - q1
    lim_inf = q1 - 1.5 * aiq
    lim_sup = q3 + 1.5 * aiq
    n_out = int(((s < lim_inf) | (s > lim_sup)).sum())
    ep = dp / np.sqrt(n)
    cv = dp / media * 100 if media != 0 else np.nan
    assim, curt = s.skew(), s.kurt()
    return {
        "n": n,
        "faltantes": total - n,
        "pct_faltantes": (total - n) / total * 100,
        "minimo": s.min(),
        "q1": q1,
        "mediana": mediana,
        "q3": q3,
        "maximo": s.max(),
        "media": media,
        "moda": s.mode().iloc[0],
        "amplitude": s.max() - s.min(),
        "aiq": aiq,
        "variancia": s.var(ddof=1),
        "desvio_padrao": dp,
        "erro_padrao": ep,
        "cv_pct": cv,
        "ic95_inf": media - 1.96 * ep,
        "ic95_sup": media + 1.96 * ep,
        "assimetria": assim,
        "curtose": curt,
        "limite_inf": lim_inf,
        "limite_sup": lim_sup,
        "n_outliers": n_out,
        "pct_outliers": n_out / n * 100,
        "classe_cv": classificar_cv(cv),
        "classe_assimetria": classificar_assimetria(assim),
        "classe_curtose": classificar_curtose(curt),
    }


est = pd.DataFrame({nome: calcular_estatisticas(df[col]) for col, nome in VARIAVEIS.items()}).T
colunas_num = [c for c in est.columns if not c.startswith("classe_")]
est[colunas_num] = est[colunas_num].astype(float)
est.shape

(12, 27)

## 2. Medidas de posição (tendência central e quartis)

In [18]:
tab_posicao = est[["n", "faltantes", "minimo", "q1", "mediana", "q3", "maximo", "media", "moda"]].copy()
tab_posicao[["n", "faltantes"]] = tab_posicao[["n", "faltantes"]].astype(int)
tab_posicao = tab_posicao.rename(columns={
    "n": "n", "faltantes": "Faltantes", "minimo": "Mín.", "q1": "Q1", "mediana": "Mediana (Q2)",
    "q3": "Q3", "maximo": "Máx.", "media": "Média", "moda": "Moda"})
display(tab_posicao.round(2))

,n,Faltantes,Mín.,Q1,Mediana (Q2),Q3,Máx.,Média,Moda
Peso ao nascer (g),142564,4,115.0,2905.0,3225.0,3530.0,7000.0,3184.02,3300.0
Idade da mãe (anos),142567,1,10.0,23.0,28.0,33.0,53.0,27.87,29.0
Semanas de gestação,138305,4263,19.0,37.0,38.0,39.0,45.0,38.09,39.0
Consultas de pré-natal,139038,3530,0.0,8.0,10.0,11.0,53.0,9.46,10.0
Apgar 1º minuto,142034,534,0.0,8.0,8.0,9.0,10.0,8.19,9.0
Apgar 5º minuto,142016,552,0.0,9.0,9.0,9.0,10.0,9.02,9.0
Gestações anteriores,130733,11835,0.0,0.0,1.0,2.0,96.0,1.34,1.0
Filhos vivos anteriores,129347,13221,0.0,0.0,1.0,2.0,30.0,1.07,0.0
Filhos mortos anteriores,122480,20088,0.0,0.0,0.0,0.0,26.0,0.31,0.0
Partos vaginais anteriores,123152,19416,0.0,0.0,0.0,1.0,77.0,0.66,0.0


## 3. Medidas de dispersão
`CV` permite comparar a variabilidade de variáveis em **unidades diferentes** (gramas × anos × semanas), pois é adimensional.

In [19]:
tab_dispersao = est[["amplitude", "aiq", "variancia", "desvio_padrao", "erro_padrao", "ic95_inf", "ic95_sup", "cv_pct", "classe_cv"]].copy()
tab_dispersao = tab_dispersao.rename(columns={
    "amplitude": "Amplitude", "aiq": "AIQ (Q3−Q1)", "variancia": "Variância", "desvio_padrao": "Desvio padrão",
    "erro_padrao": "Erro padrão", "ic95_inf": "IC95% inf.", "ic95_sup": "IC95% sup.", "cv_pct": "CV (%)", "classe_cv": "Classificação do CV"})
num = [c for c in tab_dispersao.columns if c != "Classificação do CV"]
tab_dispersao[num] = tab_dispersao[num].astype(float).round(2)
display(tab_dispersao)

,Amplitude,AIQ (Q3−Q1),Variância,Desvio padrão,Erro padrão,IC95% inf.,IC95% sup.,CV (%),Classificação do CV
Peso ao nascer (g),6885.0,625.0,321302.85,566.84,1.50,3181.07,3186.96,17.80,Média dispersão
Idade da mãe (anos),43.0,10.0,44.20,6.65,0.02,27.84,27.91,23.85,Média dispersão
Semanas de gestação,26.0,2.0,5.55,2.36,0.01,38.07,38.10,6.19,Baixa dispersão
Consultas de pré-natal,53.0,3.0,10.04,3.17,0.01,9.45,9.48,33.48,Alta dispersão
Apgar 1º minuto,10.0,1.0,1.42,1.19,0.00,8.18,8.20,14.57,Baixa dispersão
Apgar 5º minuto,10.0,0.0,0.59,0.77,0.00,9.02,9.03,8.55,Baixa dispersão
Gestações anteriores,96.0,2.0,2.04,1.43,0.00,1.33,1.35,106.46,Alta dispersão
Filhos vivos anteriores,30.0,2.0,1.38,1.17,0.00,1.07,1.08,109.42,Alta dispersão
Filhos mortos anteriores,26.0,0.0,0.40,0.64,0.00,0.31,0.32,204.00,Alta dispersão
Partos vaginais anteriores,77.0,1.0,1.34,1.16,0.00,0.65,0.66,176.27,Alta dispersão


## 4. Forma da distribuição e valores atípicos (outliers)
> **Limitação do critério de Tukey:** em variáveis discretas e muito concentradas (ex.: filhos mortos anteriores, em que Q1 = Q3 = 0 e AIQ = 0), *qualquer* valor
> diferente de zero é marcado como outlier. Nesses casos, o resultado mostra concentração em zero, não necessariamente erro de registro.

In [20]:
tab_forma = est[["assimetria", "classe_assimetria", "curtose", "classe_curtose", "limite_inf", "limite_sup", "n_outliers", "pct_outliers"]].copy()
tab_forma = tab_forma.rename(columns={
    "assimetria": "Assimetria", "classe_assimetria": "Interpretação (assimetria)", "curtose": "Curtose (excesso)",
    "classe_curtose": "Interpretação (curtose)", "limite_inf": "Limite inf. (Tukey)", "limite_sup": "Limite sup. (Tukey)",
    "n_outliers": "Nº outliers", "pct_outliers": "% outliers"})
for c in ["Assimetria", "Curtose (excesso)", "Limite inf. (Tukey)", "Limite sup. (Tukey)", "% outliers"]:
    tab_forma[c] = tab_forma[c].astype(float).round(2)
tab_forma["Nº outliers"] = tab_forma["Nº outliers"].astype(int)
display(tab_forma)

,Assimetria,Interpretação (assimetria),Curtose (excesso),Interpretação (curtose),Limite inf. (Tukey),Limite sup. (Tukey),Nº outliers,% outliers
Peso ao nascer (g),-0.95,Assimetria moderada à esquerda (negativa),3.23,Leptocúrtica,1967.5,4467.5,5224,3.66
Idade da mãe (anos),0.15,Aproximadamente simétrica,-0.65,Platicúrtica,8.0,48.0,14,0.01
Semanas de gestação,-2.21,Assimetria forte à esquerda (negativa),10.44,Leptocúrtica,34.0,42.0,6582,4.76
Consultas de pré-natal,0.63,Assimetria moderada à direita (positiva),4.91,Leptocúrtica,3.5,15.5,8616,6.20
Apgar 1º minuto,-2.84,Assimetria forte à esquerda (negativa),11.25,Leptocúrtica,6.5,10.5,8648,6.09
Apgar 5º minuto,-4.10,Assimetria forte à esquerda (negativa),35.58,Leptocúrtica,9.0,9.0,40620,28.60
Gestações anteriores,4.16,Assimetria forte à direita (positiva),162.39,Leptocúrtica,-3.0,5.0,1856,1.42
Filhos vivos anteriores,1.81,Assimetria forte à direita (positiva),8.12,Leptocúrtica,-3.0,5.0,877,0.68
Filhos mortos anteriores,3.25,Assimetria forte à direita (positiva),33.99,Leptocúrtica,0.0,0.0,29922,24.43
Partos vaginais anteriores,5.50,Assimetria forte à direita (positiva),189.98,Leptocúrtica,-1.5,2.5,8505,6.91


### 4.1 Outliers: Tukey × escore z × outliers severos
- **Tukey (1,5·AIQ):** não depende da normalidade.
- **|z| > 3:** supõe distribuição aproximadamente normal.
- **Severos (3·AIQ):** valores muito distantes dos quartis — candidatos mais fortes a erro de registro.

In [21]:
linhas = []
for col, nome in VARIAVEIS.items():
    s = df[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    aiq = q3 - q1
    z = (s - s.mean()) / s.std(ddof=1)
    linhas.append({
        "Variável": nome,
        "n": len(s),
        "Outliers Tukey (1,5·AIQ)": int(((s < q1 - 1.5 * aiq) | (s > q3 + 1.5 * aiq)).sum()),
        "Outliers severos (3·AIQ)": int(((s < q1 - 3 * aiq) | (s > q3 + 3 * aiq)).sum()),
        "|z| > 3": int((z.abs() > 3).sum()),
    })
display(pd.DataFrame(linhas).set_index("Variável"))

,n,"Outliers Tukey (1,5·AIQ)",Outliers severos (3·AIQ),|z| > 3
Variável,,,,
Peso ao nascer (g),142564,5224,1064,2165
Idade da mãe (anos),142567,14,0,33
Semanas de gestação,138305,6582,2090,2722
Consultas de pré-natal,139038,8616,380,1086
Apgar 1º minuto,142034,8648,3451,3451
Apgar 5º minuto,142016,40620,40620,1402
Gestações anteriores,130733,1856,205,1856
Filhos vivos anteriores,129347,877,83,1982
Filhos mortos anteriores,122480,29922,29922,1414


### 4.2 Valores extremos de peso e idade da mãe
Inspecione se os extremos são plausíveis. Pesos abaixo de ~500 g e acima de ~5.500 g são raros e podem indicar erro de digitação; idades de 10–14 anos existem no
SINASC, mas merecem atenção em saúde pública.

In [22]:
print("Menores pesos (g):", df["PESO"].nsmallest(8).astype(int).tolist())
print("Maiores pesos (g):", df["PESO"].nlargest(8).astype(int).tolist())
print()
print("Peso < 500 g:", int((df["PESO"] < 500).sum()), "| Peso < 1.000 g:", int((df["PESO"] < 1000).sum()),
      "| Peso > 5.000 g:", int((df["PESO"] > 5000).sum()))
print()
print("Menores idades da mãe:", df["IDADEMAE"].nsmallest(8).astype(int).tolist())
print("Maiores idades da mãe:", df["IDADEMAE"].nlargest(8).astype(int).tolist())

Menores pesos (g): [115, 135, 170, 170, 172, 185, 195, 200]
Maiores pesos (g): [7000, 7000, 7000, 7000, 6795, 6530, 6362, 6310]

Peso < 500 g: 281 | Peso < 1.000 g: 976 | Peso > 5.000 g: 91

Menores idades da mãe: [10, 11, 11, 11, 12, 12, 12, 12]
Maiores idades da mãe: [53, 53, 53, 52, 52, 51, 50, 50]


### 4.3 Sensibilidade: estatísticas **com** e **sem** outliers (Tukey)
Mostra o quanto os valores atípicos influenciam média, desvio padrão e CV. Útil para justificar no relatório por que a mediana e a AIQ são mais robustas.

In [23]:
def comparar_sem_outliers(col):
    s = df[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    aiq = q3 - q1
    limpo = s[(s >= q1 - 1.5 * aiq) & (s <= q3 + 1.5 * aiq)]
    saida = {}
    for rotulo, x in [("Com outliers", s), ("Sem outliers", limpo)]:
        saida[rotulo] = {
            "n": len(x), "média": x.mean(), "mediana": x.median(), "desvio padrão": x.std(ddof=1),
            "CV (%)": x.std(ddof=1) / x.mean() * 100, "assimetria": x.skew(), "curtose": x.kurt(),
        }
    return pd.DataFrame(saida).round(2)


for col in ["PESO", "IDADEMAE", "SEMAGESTAC"]:
    print(VARIAVEIS[col])
    display(comparar_sem_outliers(col))

Peso ao nascer (g)


,Com outliers,Sem outliers
n,142564.00,137340.00
média,3184.02,3230.51
mediana,3225.00,3240.00
desvio padrão,566.84,454.37
CV (%),17.80,14.07
assimetria,-0.95,-0.11
curtose,3.23,-0.09


Idade da mãe (anos)


,Com outliers,Sem outliers
n,142567.00,142553.00
média,27.87,27.87
mediana,28.00,28.00
desvio padrão,6.65,6.64
CV (%),23.85,23.84
assimetria,0.15,0.15
curtose,-0.65,-0.65


Semanas de gestação


,Com outliers,Sem outliers
n,138305.00,131723.00
média,38.09,38.35
mediana,38.00,38.00
desvio padrão,2.36,1.58
CV (%),6.19,4.11
assimetria,-2.21,-0.39
curtose,10.44,0.21


## 5. Percentis

In [24]:
PERCENTIS = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
tab_percentis = df[list(VARIAVEIS)].quantile(PERCENTIS).T
tab_percentis.columns = ["P" + str(int(p * 100)) for p in PERCENTIS]
tab_percentis.index = [VARIAVEIS[c] for c in tab_percentis.index]
display(tab_percentis.round(2))

,P1,P5,P10,P25,P50,P75,P90,P95,P99
Peso ao nascer (g),1235.0,2232.0,2545.0,2905.0,3225.0,3530.0,3815.0,3995.0,4368.74
Idade da mãe (anos),15.0,17.0,19.0,23.0,28.0,33.0,37.0,39.0,42.00
Semanas de gestação,29.0,34.0,36.0,37.0,38.0,39.0,40.0,41.0,42.00
Consultas de pré-natal,2.0,4.0,6.0,8.0,10.0,11.0,13.0,14.0,18.00
Apgar 1º minuto,3.0,6.0,7.0,8.0,8.0,9.0,9.0,9.0,9.00
Apgar 5º minuto,7.0,8.0,8.0,9.0,9.0,9.0,10.0,10.0,10.00
Gestações anteriores,0.0,0.0,0.0,0.0,1.0,2.0,3.0,4.0,6.00
Filhos vivos anteriores,0.0,0.0,0.0,0.0,1.0,2.0,2.0,3.0,5.00
Filhos mortos anteriores,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.00
Partos vaginais anteriores,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,5.00


## 6. Distribuições de frequência (variáveis agrupadas em classes)
`f` = frequência absoluta · `fr` = frequência relativa · `F` = acumulada · ponto médio = representante da classe. Intervalos fechados à esquerda `[a, b)`.

Referência de classes pela **regra de Sturges**: $k = 1 + 3{,}322\,\log_{10} n$.

In [25]:
def tabela_frequencia(serie, largura):
    s = serie.dropna()
    inicio = int(np.floor(s.min() / largura) * largura)
    fim = int(np.ceil((s.max() + 1) / largura) * largura)
    bins = np.arange(inicio, fim + largura, largura)
    classes = pd.cut(s, bins=bins, right=False)
    tab = pd.DataFrame({"f": classes.value_counts(sort=False)})
    tab["fr (%)"] = tab["f"] / tab["f"].sum() * 100
    tab["F"] = tab["f"].cumsum()
    tab["Fr acum. (%)"] = tab["fr (%)"].cumsum()
    tab["Ponto médio"] = [(i.left + i.right) / 2 for i in tab.index]
    tab.index = [str(int(i.left)) + " |— " + str(int(i.right)) for i in tab.index]
    return tab.round(2)


n_peso = df["PESO"].notna().sum()
k = int(np.ceil(1 + 3.322 * np.log10(n_peso)))
print("Sturges para PESO: k ≈", k, "classes (amplitude total / k ≈", round((df["PESO"].max() - df["PESO"].min()) / k), "g)")
print("Usamos classes de 500 g por legibilidade.\n")

tab_freq_peso = tabela_frequencia(df["PESO"], 500)
display(tab_freq_peso)

Sturges para PESO: k ≈ 19 classes (amplitude total / k ≈ 362 g)
Usamos classes de 500 g por legibilidade.



,f,fr (%),F,Fr acum. (%),Ponto médio
0 |— 500,281,0.20,281,0.20,250.0
500 |— 1000,695,0.49,976,0.68,750.0
1000 |— 1500,1100,0.77,2076,1.46,1250.0
1500 |— 2000,2469,1.73,4545,3.19,1750.0
2000 |— 2500,8262,5.80,12807,8.98,2250.0
2500 |— 3000,31618,22.18,44425,31.16,2750.0
3000 |— 3500,59013,41.39,103438,72.56,3250.0
3500 |— 4000,32095,22.51,135533,95.07,3750.0
4000 |— 4500,6245,4.38,141778,99.45,4250.0
4500 |— 5000,694,0.49,142472,99.94,4750.0


In [26]:
tab_freq_idade = tabela_frequencia(df["IDADEMAE"], 5)
display(tab_freq_idade)

,f,fr (%),F,Fr acum. (%),Ponto médio
10 |— 15,829,0.58,829,0.58,12.5
15 |— 20,15322,10.75,16151,11.33,17.5
20 |— 25,31243,21.91,47394,33.24,22.5
25 |— 30,37850,26.55,85244,59.79,27.5
30 |— 35,31420,22.04,116664,81.83,32.5
35 |— 40,19838,13.91,136502,95.75,37.5
40 |— 45,5716,4.01,142218,99.76,42.5
45 |— 50,339,0.24,142557,99.99,47.5
50 |— 55,10,0.01,142567,100.00,52.5


## 7. Variáveis categóricas (frequências absoluta e relativa)

In [27]:
CATEGORICAS = {
    "SEXO_DESC": "Sexo",
    "PARTO_DESC": "Tipo de parto",
    "GRAVIDEZ_DESC": "Tipo de gravidez",
    "GESTACAO_DESC": "Duração da gestação",
    "RACACOR_DESC": "Raça/cor do recém-nascido",
    "ESCMAE_DESC": "Escolaridade da mãe",
    "ANO": "Ano de nascimento",
}

tabelas_cat = {}
for col, nome in CATEGORICAS.items():
    f = df[col].value_counts(dropna=False).sort_index() if col == "ANO" else df[col].value_counts(dropna=False)
    t = pd.DataFrame({"f": f, "fr (%)": (f / f.sum() * 100).round(2)})
    tabelas_cat[nome] = t
    print(nome)
    display(t)

Sexo


,f,fr (%)
SEXO_DESC,,
Masculino,72859,51.10
Feminino,69684,48.88
Ignorado,25,0.02


Tipo de parto


,f,fr (%)
PARTO_DESC,,
Cesáreo,98728,69.25
Vaginal,43639,30.61
Não informado,197,0.14
Ignorado,4,0.00


Tipo de gravidez


...

,f,fr (%)
RACACOR_DESC,,
Parda,108025,75.77
Branca,15962,11.20
Não informado,13994,9.82
Preta,3648,2.56
Indígena,543,0.38
Amarela,396,0.28


Escolaridade da mãe


,f,fr (%)
ESCMAE_DESC,,
8 a 11 anos,96259,67.52
12 anos ou mais,29811,20.91
4 a 7 anos,13801,9.68
1 a 3 anos,1264,0.89
Não informado,875,0.61
Ignorado,322,0.23
Nenhuma,236,0.17


Ano de nascimento


,f,fr (%)
ANO,,
2025,107346,75.29
2026,35222,24.71


In [28]:
indicadores = df[["BAIXO_PESO", "PREMATURO", "CESAREO"]].mean() * 100
indicadores.index = ["Baixo peso (< 2.500 g)", "Prematuridade (< 37 sem)", "Parto cesáreo"]
print("Indicadores gerais (% entre os registros com informação):")
display(indicadores.round(2).to_frame("%"))

Indicadores gerais (% entre os registros com informação):


,%
Baixo peso (< 2.500 g),8.98
Prematuridade (< 37 sem),14.48
Parto cesáreo,69.35


## 8. Texto-modelo para o relatório
Gera um parágrafo-base por variável. **Revise e adapte** a redação antes de colar no relatório.

In [29]:
def br(x, casas=1):
    # formata número no padrão brasileiro: 1.234,5
    t = "{:,.{c}f}".format(x, c=casas)
    return t.replace(",", "#").replace(".", ",").replace("#", ".")


def texto_variavel(nome, r):
    casas = 0 if r["media"] >= 100 else 1
    return (
        nome + ": média de " + br(r["media"], casas) + " (DP = " + br(r["desvio_padrao"], casas + 1) + "; CV = " + br(r["cv_pct"]) + "%, "
        + r["classe_cv"].lower() + "), mediana de " + br(r["mediana"], casas) + ". Metade central dos valores entre "
        + br(r["q1"], casas) + " e " + br(r["q3"], casas) + " (AIQ = " + br(r["aiq"], casas) + "); mínimo de " + br(r["minimo"], casas)
        + " e máximo de " + br(r["maximo"], casas) + ". Distribuição: " + r["classe_assimetria"].lower() + " (g1 = " + br(r["assimetria"], 2)
        + ") e " + r["classe_curtose"].lower() + " (curtose = " + br(r["curtose"], 2) + "). Foram identificados " + br(r["n_outliers"], 0)
        + " valores atípicos pelo critério de Tukey (" + br(r["pct_outliers"]) + "%), com n = " + br(r["n"], 0) + "."
    )


for nome, linha in est.iterrows():
    print(texto_variavel(nome, linha))
    print()

Peso ao nascer (g): média de 3.184 (DP = 566,8; CV = 17,8%, média dispersão), mediana de 3.225. Metade central dos valores entre 2.905 e 3.530 (AIQ = 625); mínimo de 115 e máximo de 7.000. Distribuição: assimetria moderada à esquerda (negativa) (g1 = -0,95) e leptocúrtica (curtose = 3,23). Foram identificados 5.224 valores atípicos pelo critério de Tukey (3,7%), com n = 142.564.

Idade da mãe (anos): média de 27,9 (DP = 6,65; CV = 23,9%, média dispersão), mediana de 28,0. Metade central dos valores entre 23,0 e 33,0 (AIQ = 10,0); mínimo de 10,0 e máximo de 53,0. Distribuição: aproximadamente simétrica (g1 = 0,15) e platicúrtica (curtose = -0,65). Foram identificados 14 valores atípicos pelo critério de Tukey (0,0%), com n = 142.567.

Semanas de gestação: média de 38,1 (DP = 2,36; CV = 6,2%, baixa dispersão), mediana de 38,0. Metade central dos valores entre 37,0 e 39,0 (AIQ = 2,0); mínimo de 19,0 e máximo de 45,0. Distribuição: assimetria forte à esquerda (negativa) (g1 = -2,21) e lept

## 9. Exportação das tabelas
Os CSV usam `;` e vírgula decimal (abrem direto no Excel em português). Se tiver `openpyxl`, gera também um `.xlsx` com todas as abas.

In [30]:
tabelas_export = {
    "estat_posicao": tab_posicao.round(2),
    "estat_dispersao": tab_dispersao,
    "estat_forma_outliers": tab_forma,
    "percentis": tab_percentis.round(2),
    "freq_peso": tab_freq_peso,
    "freq_idade_mae": tab_freq_idade,
}

for nome, tabela in tabelas_export.items():
    tabela.to_csv(PASTA_TABELAS / (nome + ".csv"), sep=";", decimal=",", encoding="utf-8-sig")
try:
    with pd.ExcelWriter(PASTA_TABELAS / "estatisticas_descritivas.xlsx") as writer:
        for nome, tabela in tabelas_export.items():
            tabela.to_excel(writer, sheet_name=nome[:31])
        for nome, tabela in tabelas_cat.items():
            # Substitui a barra por um hífen para respeitar as regras do Excel
            nome_aba = ("cat_" + nome.replace("/", "-"))[:31] 
            tabela.to_excel(writer, sheet_name=nome_aba)
    print("Excel gerado com sucesso.")
except ImportError:
    print("openpyxl não instalado: apenas os CSV foram gerados (pip install openpyxl).")

Excel gerado com sucesso.
